# Zwierlein lab setup

```bash
uv sync --extra hardware
```

The `hardware` extra installs `spcm` (and on Linux, CUDA). See [setup_guide.md](setup_guide.md).


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from atommovr.utils.core import Configurations, PhysicalParams
from atommovr.utils.errormodels import UniformVacuumTweezerError, ZeroNoise
from atommovr.utils.timing import all_phase_duration_s, travel_duration_s
from aod_atommovr import (
    AodController,
    GaussianCameraConfig,
    HardwareConfig,
    OfflineArrayCamera,
    RealArrayCamera,
    SoftwareConfig,
)
from awg_controller import AODSettings, RFConverter
from recorder import Recorder

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
)

## 1. Lab params


In [ ]:
ROWS, COLS = 30, 30

print(f"lattice = {ROWS}x{COLS}")

# Only used for simulation mode (synced with OfflineArrayCamera)
lab_error = UniformVacuumTweezerError(
    pickup_time=0.1e-6,  # s
    putdown_time=0.1e-6,  # s
    accel_time=0.1e-6,  # s
    decel_time=0.1e-6,  # s
    pickup_fail_rate=0.01,
    putdown_fail_rate=0.01,
    lifetime=5e3,  # s
    seed=0,
)

lab_params = PhysicalParams(
    spacing=8e-6,  # m
    AOD_speed=3,  # µm/µs m/s
    loading_prob=0.65,  # enough atoms for a middle target
    middle_size=[20, 20],  # must fit inside ROWS x COLS
)


class _FakeMove:
    def __init__(self, fr, fc, tr, tc):
        self.from_row, self.from_col, self.to_row, self.to_col = fr, fc, tr, tc


# travel_duration_s floors every move batch at MIN_MOVE_DURATION_S (5 us,
# atommovr/utils/timing.py); shorter raw Chebyshev-distance travel times
# are floored up to that value.
travel_s = travel_duration_s(
    [_FakeMove(0, 0, 3, 2)], lab_params.spacing, lab_params.AOD_speed
)
phase_s = all_phase_duration_s(lab_error)

print(
    f"travel (3x2 Chebyshev) = {travel_s * 1e6:.1f}µs; all phases = {phase_s * 1e6:.1f}µs"
)

## 2. AOD settings


In [ ]:
# Internally full grid size is based on AOD settings
aod = AODSettings(
    grid_rows=ROWS,
    grid_cols=COLS,
    f_min_v=85e6,
    f_max_v=121e6,
    f_min_h=85.5e6,
    f_max_h=121.5e6,
    alignment="center",
)

bw_v_mhz = (aod.f_max_v - aod.f_min_v) / 1e6
bw_h_mhz = (aod.f_max_h - aod.f_min_h) / 1e6
print(f"lattice = {aod.grid_rows}x{aod.grid_cols}")
print(f"Δf_v = {aod.f_spacing_v / 1e6:.3f} MHz/site (bandwidth {bw_v_mhz:.1f} MHz)")
print(f"Δf_h = {aod.f_spacing_h / 1e6:.3f} MHz/site (bandwidth {bw_h_mhz:.1f} MHz)")

# RF tone converter - uses spacing for timing used to pause AWG
rf = RFConverter(aod, lab_params)
row_idx = np.arange(aod.grid_rows)
freqs = [rf._row_to_freq(i) for i in row_idx]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(row_idx, np.asarray(freqs) / 1e6)
ax.set_xlabel("row index")
ax.set_ylabel("RF frequency (MHz)")
ax.grid(True, alpha=0.3)
plt.show()

## 3. Physical spacing

Theoretical slope: $dx/df = f_\mathrm{obj}\,(f_1/f_2)\,(\lambda/v)$.


In [ ]:
wavelength_m, v_acoustic = 808e-9, 650.0
f1_mm, f2_mm, f_obj_mm = 75.0, 400.0, 28.0
um_per_mhz = (
    (f_obj_mm * 1e-3) * (f1_mm / f2_mm) * (wavelength_m / v_acoustic) * 1e12
)  # m/Hz -> µm/MHz
aod_bandwidth_mhz = bw_v_mhz
fov_um = um_per_mhz * aod_bandwidth_mhz

print(f"um_per_mhz = {um_per_mhz:.3f}")
print(
    f"FOV = {fov_um:.1f} µm over {aod_bandwidth_mhz:.0f} MHz (fov_um_v={aod.fov_um_v:.1f})"
)
print(
    f"theoretical spacing_x = {fov_um/(aod.grid_cols-1):.2f} µm, spacing_y = {fov_um/(aod.grid_rows-1):.2f} µm"
)

aod.um_per_mhz = um_per_mhz  # not used internally, just a helper
lab_params.spacing = fov_um / (aod.grid_cols - 1) * 1e-6  # update spacing to match fov

## 4. Camera

Allied Vision Alvium 1800 U-052: 816×624, 8-bit mono

Both `OfflineArrayCamera` and `RealArrayCamera` inherit `Camera.detect_occupancy` (blob -> rotate -> `fit_grid_and_assign`). Stage dumps happen when the controller calls `camera.sync(array)`


In [ ]:
# Gaussian image configuration
lab_camera = GaussianCameraConfig(
    image_shape=(624, 816),  # (H, W)
    sigma_px=2,
    peak_counts=180.0,
    background=12.0,  # residual scatter to minimize optically
    noise_level=5,
    stripe_intensity=1,
    min_spacing_px=15,
    spacing_x=20,  # px
    spacing_y=20,  # px
    angle=1,  # camera angle
    dtype=np.uint8,  # 8-bit mono
)

# Fake atom array generation and simulation
offline_preview = OfflineArrayCamera(
    (ROWS, COLS),
    image_generator=lab_camera,
    physical_params=PhysicalParams(loading_prob=0.6, spacing=1.00e-6),
    seed=42,
)

frame = offline_preview.acquire()
occ_det = offline_preview.detect_occupancy(frame)
print(f"frame shape={frame.shape}, dtype={frame.dtype}, max={frame.max()}")
print(
    f"occupancy (truth) = {offline_preview.occupancy.sum()} / {ROWS * COLS}, "
    f"detected = {int(occ_det.sum())} / {ROWS * COLS}"
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(frame, cmap="gray", origin="upper")
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
plt.show()

## 5. Offline test

Uses `AodController(..., camera=offline_cam, hooks=[recorder])`.

- `hooks.on_session_start` writes `meta.json`
- Each round: `camera.sync(array)`, then move / RF stats append to `rounds.jsonl` from `hooks.after_round`

Pick an algorithm + target pattern in the cell below

In [ ]:
from atommovr.utils.core import Configurations

ALGORITHM_NAME = "Hungarian"
# ALGORITHM_NAME options: "PCFA", "Hungarian", "Tetris", "BalanceAndCompact",
#                          "BCv2", "ParallelLBAP", "ParallelHungarian", "GeneralizedBalance"

TARGET_TYPE = Configurations.MIDDLE_FILL
# TARGET_TYPE options: Configurations.MIDDLE_FILL, .ZEBRA_HORIZONTAL, .ZEBRA_VERTICAL,
#                       .CHECKERBOARD, .Left_Sweep, .RANDOM

print(f"algorithm = {ALGORITHM_NAME!r}")
print(f"target pattern = {TARGET_TYPE.name}")

In [ ]:
grid = (aod.grid_rows, aod.grid_cols)

# For data storage
recorder = Recorder(
    "runs",
    meta={
        "grid": list(grid),
        "target": list(lab_params.middle_size),
        "algo": ALGORITHM_NAME,
        "seed": 0,
        "note": "Offline test",
    },
)

offline_cam = OfflineArrayCamera(
    grid,
    image_generator=lab_camera,
    physical_params=lab_params,
    seed=0,
)

# Information needed for algorithm run
sw = SoftwareConfig(
    algorithm_name=ALGORITHM_NAME,
    max_rounds=5,
    target_type=TARGET_TYPE,
    error_model=lab_error,  # Only needed for simulation mode
)

# AWG settings -- HardwareConfig owns aod_settings/physical_params, not SoftwareConfig
hw = HardwareConfig(
    max_amplitude_v=1.6, aod_settings=aod, physical_params=lab_params
)  # sim

with AodController(sw, hw, camera=offline_cam, hooks=[recorder]) as ctrl:
    ok = ctrl.run()
    mask = (ctrl.array.target[:, :, 0] > 0).astype(int)
    occ = offline_cam.occupancy

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(mask, cmap="Blues", origin="upper", vmin=0, vmax=1)
axes[0].set_title(f"Target ({TARGET_TYPE.name})")
if occ is not None:
    axes[1].imshow(occ, cmap="Blues", origin="upper")
    filled = int((occ * mask).sum())
    axes[1].set_title(f"Final occ ({filled}/{int(mask.sum())} target sites)")
print(f"success={ok}")

print(f"check {recorder.run_dir}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(mask, cmap="Blues", origin="upper", vmin=0, vmax=1)
axes[0].set_title(f"Target ({TARGET_TYPE.name})")
if occ is not None:
    axes[1].imshow(occ, cmap="Blues", origin="upper")
    filled = int((occ * mask).sum())
    axes[1].set_title(f"Final occ ({filled}/{int(mask.sum())} target sites)")
print(f"success={ok}")

print(f"check {recorder.run_dir}")
plt.tight_layout()
plt.show()

## 6. Hardware control (self-contained)

End-to-end on the apparatus:

```text
Alvium (or OfflineArrayCamera)
        │ acquire()
        ▼
Camera.detect_occupancy  ->  occupancy grid
        │
        ▼
Algorithm (Hungarian / ...)  ->  move batches
        │
        ▼
RFConverter  ->  AWGBatch (RFRamp list)
        │
        ▼
AWGEngine  ->  spcm.SCAPPTransfer + GPU
        ▼
Spectrum AWG  →  AOD RF
```

Safety (do this before connecting the AOD amp):

- `HardwareConfig.max_amplitude_v` must stay <= 2.0 V (default 1.6 V). Start at 1.0 V and check on a scope.
- `AWGEngine` needs `spcm` and `cupy`/CUDA to touch real hardware; missing either runs this cell in simulation.
- `AodController` is not wired up to `AWGEngine` yet, so this cell always runs in simulation regardless of `hw`.
- Move-batch visualization (if enabled) runs in `after_round` and adds wall-clock latency on real hardware.


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

from atommovr.utils.core import Configurations, PhysicalParams
from atommovr.utils.errormodels import UniformVacuumTweezerError, ZeroNoise
from atommovr.utils.timing import all_phase_duration_s, travel_duration_s
from aod_atommovr import (
    AodController,
    GaussianCameraConfig,
    HardwareConfig,
    OfflineArrayCamera,
    RealArrayCamera,
    SoftwareConfig,
)
from awg_controller import AODSettings, RFConverter
from recorder import Recorder

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
)

In [ ]:
ROWS, COLS = 30, 30

# AOD
aod = AODSettings(
    grid_rows=ROWS,
    grid_cols=COLS,
    f_min_v=85e6,
    f_max_v=121e6,
    f_min_h=85.5e6,
    f_max_h=121.5e6,
    alignment="center",
)

# Physical params
lab_params = PhysicalParams(
    spacing=8.10e-6,  # m
    AOD_speed=3,  # µm/µs m/s
    loading_prob=0.65,  # enough atoms for a middle target
    middle_size=[20, 20],  # must fit inside ROWS x COLS
)

# AWG -- HardwareConfig owns aod_settings/physical_params, not SoftwareConfig
hw = HardwareConfig(
    card_path="/dev/spcm0",  # macOS/Linux device node (single card)
    max_amplitude_v=1.0,  # start conservative; raise toward <=1.6 after scope check
    output_load_ohms=50.0,
    aod_settings=aod,
    physical_params=lab_params,
)

# RF converter
rf = RFConverter(aod, lab_params)

# Algorithm
ALGORITHM_NAME = "BCv2"
# ALGORITHM_NAME options: "PCFA", "Hungarian", "Tetris", "BalanceAndCompact",
#                          "BCv2", "ParallelLBAP", "ParallelHungarian", "GeneralizedBalance"

TARGET_TYPE = Configurations.MIDDLE_FILL
# TARGET_TYPE options: Configurations.MIDDLE_FILL, .ZEBRA_HORIZONTAL, .ZEBRA_VERTICAL,
#                       .CHECKERBOARD, .Left_Sweep, .RANDOM

sw = SoftwareConfig(
    max_rounds=5,
    algorithm_name=ALGORITHM_NAME,
    target_type=TARGET_TYPE,
    # error_model=lab_error,
)

# Data
recorder = Recorder(
    "runs",
    meta={
        "grid": [aod.grid_rows, aod.grid_cols],
        "target": list(lab_params.middle_size),
        "algo": ALGORITHM_NAME,
        "seed": 0,
        "note": "Test",
    },
)

lab_camera = GaussianCameraConfig(
    image_shape=(624, 816),  # (H, W)
    sigma_px=2,
    peak_counts=180.0,
    background=12.0,  # residual scatter to minimize optically
    noise_level=5,
    stripe_intensity=1,
    min_spacing_px=15,
    spacing_x=20,  # px
    spacing_y=20,  # px
    angle=1,  # camera angle
    dtype=np.uint8,  # 8-bit mono
)

# Reset offline camera
offline_cam = OfflineArrayCamera(
    grid_shape=(aod.grid_rows, aod.grid_cols),
    image_generator=lab_camera,
    physical_params=lab_params,
    seed=0,
)


# Uncomment when using actual camera
# def alvium_grab() -> np.ndarray:
#     """Placeholder: return one mono frame from the lab camera."""
#     raise NotImplementedError("hook up the Alvium SDK here")


# def lab_camera_fn() -> np.ndarray:
#     """Callable for RealArrayCamera"""
#     try:
#         return alvium_grab()
#     except NotImplementedError:
#         return offline_cam.acquire()


# real_cam = RealArrayCamera((aod.grid_rows, aod.grid_cols), camera_fn=lab_camera_fn)
# frame_hw = real_cam.acquire()
# occ_hw = real_cam.detect_occupancy(frame_hw)
# print(f"RealArrayCamera frame={frame_hw.shape}, detected atoms={int(occ_hw.sum())}")

In [ ]:
with AodController(
    sw, hw, camera=offline_cam, hooks=[recorder]
) as ctrl:  # change camera to real_cam
    ok = ctrl.run()

print(f"success={ok}")

# AodController is not wired to AWGEngine yet; this always runs in simulation
# (see aod_atommovr.controller module docstring).

In [ ]:
import json as _json

_rounds = recorder.run_dir / "rounds.jsonl"
if _rounds.is_file():
    _rows = [_json.loads(l) for l in _rounds.read_text().splitlines() if l.strip()]
    _move_rows = [r for r in _rows if r.get("n_moves", 0) > 0]
    for i, _move_row in enumerate(_move_rows):
        # renamed from rf_duration_s; fall back so runs recorded before the
        # rename still read back.
        _travel_s = _move_rows[-1].get(
            "total_travel_duration_s", _move_rows[-1].get("rf_duration_s")
        )
        print(
            f"{i+1}-round: n_moves={_move_rows[-1]['n_moves']}, "
            f"rf_batches={_move_rows[-1].get('n_rf_batches')}, "
            f"total_travel_duration_s={_travel_s}"
        )
        print(f"  moves sample: {_move_rows[-1].get('moves', [])[:4]}")

### CLI

With `spcm` + `cupy`/CUDA installed and the card present (`scapp`, default):

```bash
python -m aod_atommovr.controller \
  --algorithm Hungarian \
  --grid-rows 14 --grid-cols 14 \
  --target-rows 10 --target-cols 10 \
  --f-min-v 82e6 --f-max-v 118e6 \
  --f-min-h 82e6 --f-max-h 118e6 \
  --card /dev/spcm0
```
